# Agent Evaluation, Observability, Cost, and Latency

> **The story.** Traditional model evaluation scores an output. Agent evaluation must also score the route, tool names, arguments, state transitions, retries, and cost that produced it.
>
> **Where you are.** OrderFlow's demo returns “completed,” yet Elena cannot tell whether a regression came from routing or a tool argument. This chapter turns the workflow into a traceable test subject.
>
> **Notation.** $y$ is final task outcome; $	au=(e_1,...,e_n)$ is the trajectory; $L_i$ and $C_i$ are latency and cost for event $i$; $T_{expected}$ is the expected tool sequence.

## 0 - The Challenge

> **The mission:** detect every seeded route and tool regression, localize it to one trace event, and attribute cost and latency per step without a live model.

```mermaid
flowchart LR
    A["Completed answer"] --> B["Looks healthy"]
    B --> C["Hidden bad route or args"]
    C --> D["Trace spans + trajectory graders"]
    D --> E["Localized regression"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parent
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from copy import deepcopy

from shared import SUPPLIER_QUOTES, TraceRecorder, approval_route, request_by_id

EVAL_CASES = [
    {"request": request_by_id("PO-7293"), "expected_tools": ["parse", "inventory", "quote", "approve"], "expected_route": "auto"},
    {"request": request_by_id("PO-7308"), "expected_tools": ["parse", "inventory", "quote", "approve"], "expected_route": "manager"},
    {"request": request_by_id("PO-7307"), "expected_tools": ["parse", "inventory", "quote", "approve"], "expected_route": "finance"},
]
print(f"Loaded {len(EVAL_CASES)} versioned evaluation cases.")


## 1 - Failure First: Final Status Hides Bad Trajectories

A controller can end at `completed` after choosing the wrong route or passing the wrong quantity. Final-answer scoring sees the same terminal string.

```mermaid
flowchart TD
    F["Final status: completed"] --> R["Route bug"]
    F --> A["Argument bug"]
    R --> X["Invisible to final-only grader"]
    A --> X
    style F fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build a traceable deterministic workflow with injectable bugs --------
def run_workflow(request, bug=None):
    recorder = TraceRecorder(trace_id=f"trace-{request['request_id']}-{bug or 'clean'}")
    parsed_quantity = request["quantity"] + 1 if bug == "tool_argument" else request["quantity"]
    recorder.record("intake", "parse", "ok", latency_ms=8, token_count=30, cost_units=0.2, details={"quantity": parsed_quantity})
    recorder.record("inventory", "inventory", "ok", latency_ms=35, cost_units=0.5, details={"sku": request["sku"], "quantity": parsed_quantity})
    quote = min(
        [q for q in SUPPLIER_QUOTES[request["sku"]] if q["trusted"] and q["age_hours"] <= 48],
        key=lambda item: item["unit_price"],
    )
    recorder.record("supplier", "quote", "ok", latency_ms=quote["delay_ms"], cost_units=0.8, details={"supplier": quote["supplier"], "unit_price": quote["unit_price"]})
    total = quote["unit_price"] * parsed_quantity
    route = "auto" if bug == "route" else approval_route(total)
    recorder.record("finance", "approve", "ok", latency_ms=20, cost_units=0.4, details={"route": route, "total": total})
    return {"status": "completed", "route": route, "trace": recorder.as_dicts(), "totals": recorder.totals()}

clean = run_workflow(EVAL_CASES[0]["request"])
route_bug = run_workflow(EVAL_CASES[2]["request"], bug="route")
argument_bug = run_workflow(EVAL_CASES[0]["request"], bug="tool_argument")
assert clean["status"] == route_bug["status"] == argument_bug["status"] == "completed"
print("Final-only grader sees all three runs as completed.")


## 2 - Trace Spans and Correlation IDs

Each event needs one trace ID, ordered step number, actor, action, outcome, latency, cost, and the arguments needed for replay. Logs answer “what happened”; traces answer “where in this request did it happen?”

```mermaid
flowchart LR
    T["trace_id"] --> S1["parse span"]
    S1 --> S2["inventory span"]
    S2 --> S3["quote span"]
    S3 --> S4["approval span"]
    style T fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S1 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S2 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S3 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S4 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Define component, trajectory, and state-transition graders ----------
def grade_run(run, case):
    actions = [event["action"] for event in run["trace"]]
    parsed_quantity = run["trace"][0]["details"]["quantity"]
    failures = []
    if actions != case["expected_tools"]:
        failures.append({"metric": "tool_sequence", "step": None, "actual": actions})
    if parsed_quantity != case["request"]["quantity"]:
        failures.append({"metric": "argument_accuracy", "step": 1, "actual": parsed_quantity})
    if run["route"] != case["expected_route"]:
        failures.append({"metric": "approval_route", "step": 4, "actual": run["route"]})
    return {"passed": not failures, "failures": failures}

clean_grade = grade_run(clean, EVAL_CASES[0])
route_grade = grade_run(route_bug, EVAL_CASES[2])
argument_grade = grade_run(argument_bug, EVAL_CASES[0])
print("Clean:", clean_grade)
print("Route bug:", route_grade)
print("Argument bug:", argument_grade)
assert clean_grade["passed"]
assert route_grade["failures"] == [{"metric": "approval_route", "step": 4, "actual": "auto"}]
assert argument_grade["failures"][0]["step"] == 1


## 3 - Cost and Latency Attribution

Aggregate latency tells you the request is slow. Per-step attribution tells you which dependency to fix. Use percentiles for a fixture suite; averages hide the tail.

```mermaid
flowchart LR
    E["Trace events"] --> L["Latency by action"]
    E --> C["Cost by action"]
    L --> P["p50 / p95"]
    C --> B["Budget gate"]
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Attribute the clean suite and expose the dominant dependency --------
from shared import percentile

suite_runs = [run_workflow(case["request"]) for case in EVAL_CASES]
latencies = [run["totals"]["latency_ms"] for run in suite_runs]
costs = [run["totals"]["cost_units"] for run in suite_runs]
action_latency = {}
for run in suite_runs:
    for event in run["trace"]:
        action_latency[event["action"]] = action_latency.get(event["action"], 0) + event["latency_ms"]

slowest_action = max(action_latency, key=action_latency.get)
print("Latency by action:", action_latency)
print(f"p95 request latency: {percentile(latencies, 0.95):.0f} ms")
print(f"Cost per request: {costs}")
print("Dominant latency source:", slowest_action)
assert slowest_action == "quote" and all(cost <= 2.0 for cost in costs)


## 4 - Regression Harness and Deterministic Replay

A release suite should run without a provider, compare actual trajectories with versioned expectations, and emit one pass/fail record per seeded defect.

```mermaid
flowchart TD
    D["Versioned cases"] --> R["Deterministic replay"]
    R --> G["Trajectory graders"]
    G -->|"Pass"| P["Release candidate"]
    G -->|"Fail"| F["Localized defect"]
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Seed regressions and prove the harness detects every one ------------
seeded = [
    ("route", EVAL_CASES[2], run_workflow(EVAL_CASES[2]["request"], bug="route")),
    ("tool_argument", EVAL_CASES[0], run_workflow(EVAL_CASES[0]["request"], bug="tool_argument")),
]
report = []
for bug_name, case, run in seeded:
    grade = grade_run(run, case)
    report.append({"bug": bug_name, "detected": not grade["passed"], "failures": grade["failures"]})

print(json.dumps(report, indent=2))
assert all(row["detected"] for row in report)
assert {row["failures"][0]["step"] for row in report} == {1, 4}
print("PASS: every seeded regression was detected and localized.")


## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Regressions localized"] --> B["Next: governance"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After |
|---|---:|---:|
| Seeded regression detection | 0/2 with final-only scoring | 2/2 |
| Defect localization | Unknown | Exact step 1 or 4 |
| Cost attribution | Request total only | Per event and request |
| Latency attribution | Average only | Per action plus p95 |

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Traces, spans, trajectory graders, state checks, cost, latency, replay |
| Explained and illustrated | Offline versus online evaluation and evaluator agreement |
| Named with a reason | LLM-as-judge, deferred because deterministic ground truth is available |

### Key Takeaways

- Final answers cannot grade the route that produced them.
- Every side effect needs a trace ID and replayable arguments.
- Attribute latency and cost to steps before optimizing totals.
- A useful regression report points to the broken transition.
